# NSL-KDD focal-loss MLP benchmark


### Connexion && Synchronisation Drive - Github

In [ ]:
try:
    import google.colab
    google.colab.drive.mount('/content/drive')
except Exception:
    pass

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/learning-cifar10-from-scratch')
print(PROJECT_ROOT.exists())

DATA_ROOT = PROJECT_ROOT / 'data' / 'nsl-kdd'
MODEL_ROOT = PROJECT_ROOT / 'models' / 'nsl-kdd'
COLAB_ROOT = Path('/content/learning-cifar10-from-scratch')
REPO_URL = 'https://github.com/Appolinairee/-research-scratch-deep-learning-vision.git'

for path in [DATA_ROOT, MODEL_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

In [ ]:
import subprocess

if not (PROJECT_ROOT / '.git').exists():
    # Clone dans un tmp puis déplace le .git
    TMP = Path('/content/repo_tmp')
    subprocess.run(['rm', '-rf', str(TMP)], check=True)
    subprocess.run(['git', 'clone', REPO_URL, str(TMP)], check=True)
    # Déplace uniquement le contenu (pas data/ et models/)
    subprocess.run([
        'rsync', '-av', '--exclude=data/', '--exclude=models/',
        f'{TMP}/', str(PROJECT_ROOT),
    ], check=True)
    subprocess.run(['rm', '-rf', str(TMP)], check=True)
else:
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'remote', 'set-url', 'origin', REPO_URL],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'checkout', 'main'],
        check=True,
    )
    subprocess.run([
        'find', str(PROJECT_ROOT), '-type', 'd', '-name', '__pycache__',
        '-prune', '-exec', 'rm', '-rf', '{}', '+',
    ], check=True)
    pull = subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', 'main'],
        capture_output=True,
        text=True,
    )
    if pull.returncode != 0:
        raise RuntimeError(
            'Synchronisation Git impossible.\n'
            + pull.stderr
            + '\nVérifie les changements locaux et la divergence de la branche main.'
        )
    print(pull.stdout.strip())
    subprocess.run(
        [
            'git', '-C', str(PROJECT_ROOT), 'restore',
            '--source=origin/main', '--',
            'src/models/__init__.py',
            'src/models/sequential.py',
            'src/activations/__init__.py',
            'src/activations/dropout.py',
            'src/data/__init__.py',
            'src/data/augmentation.py',
            'src/optim/__init__.py',
            'src/optim/scheduler.py',
            'notebooks/models/nonlinear_binary_classifier.ipynb',
        ],
        check=True,
    )
    status = subprocess.run(
        [
            'git', '-C', str(PROJECT_ROOT), 'status', '--short',
            '--untracked-files=all', '--',
            ':(exclude)data/**', ':(exclude)models/**',
            ':(exclude)**/__pycache__/**', ':(exclude)**/*.pyc',
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    if status.stdout.strip():
        raise RuntimeError(
            'Le dépôt Drive contient des changements locaux après synchronisation :\n'
            + status.stdout
        )

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/learning-cifar10-from-scratch')
DATA_ROOT = PROJECT_ROOT / 'data' / 'nsl-kdd'
MODEL_ROOT = PROJECT_ROOT / 'models' / 'nsl-kdd'
COLAB_ROOT = Path('/content/learning-cifar10-from-scratch')
REPO_URL = 'https://github.com/Appolinairee/-research-scratch-deep-learning-vision.git'

for path in [DATA_ROOT, MODEL_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

In [ ]:
import sys
from pathlib import Path

# Ensure PROJECT_ROOT is defined; if not, find repo root by locating 'src' folder
try:
    PROJECT_ROOT
except NameError:
    repo_root = Path.cwd()
    while not (repo_root / "src").exists() and repo_root.parent != repo_root:
        repo_root = repo_root.parent
    PROJECT_ROOT = repo_root

# On Colab we may want a convenience symlink, but it's optional and may fail due to permissions
if 'google.colab' in sys.modules:
    COLAB_ROOT = Path('/content/learning-cifar10-from-scratch')
    try:
        if not COLAB_ROOT.exists():
            COLAB_ROOT.symlink_to(PROJECT_ROOT)
    except Exception as e:
        print('Could not create symlink:', e)

# Verify synchronization before importing project modules.
backend_path = PROJECT_ROOT / 'src' / 'core' / 'backend.py'
if not backend_path.is_file():
    raise RuntimeError(
        f'Code Drive obsolète : fichier manquant {backend_path}. '
        'Exécute d’abord les cellules de synchronisation Git (3-4), puis redémarre le kernel.'
    )

# Ensure the project root is on sys.path so `import src...` works
for module_name in list(sys.modules):
    if module_name == 'src' or module_name.startswith('src.'):
        del sys.modules[module_name]

sys.path[:] = [entry for entry in sys.path if entry != str(PROJECT_ROOT)]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print('Added to sys.path:', PROJECT_ROOT)
else:
    print('Project root already on sys.path:', PROJECT_ROOT)

## Installer les dependances

In [ ]:
import subprocess

# Keep exactly one CuPy CUDA wheel installed in the Colab runtime.
subprocess.run([
    'pip', 'uninstall', '-y', 'cupy-cuda12x', 'cupy-cuda13x',
    '--break-system-packages',
], check=True)
subprocess.run([
    'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements-colab.txt'),
    '--break-system-packages',
], check=True)

**But** : charger le notebook, les outils numeriques et le framework maison.

**Pourquoi** : garantir qu'on benchmarke notre propre logique, pas un modele externe.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score, precision_score, recall_score

from src.activations import Dropout, ReLU, Softmax
from src.layers import Dense
from src.losses import MulticlassFocalLoss
from src.models import Sequential
from src.optim import ReduceLROnPlateau, SGD
from src.regularization import L2Regularization
from src.training.sequential import train_multiclass_classifier

from src.evaluation.classification import evaluate_multiclass
np.set_printoptions(precision=4, suppress=True)

In [ ]:
subprocess.run(
    ['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', 'main'],
    check=True,
)

local_commit = subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'],
    text=True,
).strip()
remote_commit = subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'origin/main'],
    text=True,
).strip()

print('Drive :', local_commit)
print('GitHub:', remote_commit)
print('Aligné :', local_commit == remote_commit)


## NSL-KDD - Network intrusion benchmark

Charger NSL-KDD et encoder ses 41 variables réseau pour le MLP.

**Pourquoi** : reproduire le benchmark DNN de l’article avec notre implémentation dense from scratch.

In [ ]:
import csv
import urllib.request

NSL_KDD_URLS = {
    'train': 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt',
    'test': 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt',
}
NSL_KDD_ROOT = DATA_ROOT
NSL_KDD_ROOT.mkdir(parents=True, exist_ok=True)

for split, url in NSL_KDD_URLS.items():
    destination = NSL_KDD_ROOT / f'KDD{split.title()}%2B.txt'
    if not destination.exists():
        urllib.request.urlretrieve(url, destination)

NSL_KDD_ROOT

# Les catégories d’attaque sont regroupées selon les 5 classes de l’article.
ATTACK_GROUPS = {
    'normal': 'normal',
    'back': 'dos', 'land': 'dos', 'neptune': 'dos', 'pod': 'dos',
    'smurf': 'dos', 'teardrop': 'dos', 'mailbomb': 'dos', 'apache2': 'dos',
    'processtable': 'dos', 'udpstorm': 'dos', 'worm': 'dos',
    'ipsweep': 'probe', 'nmap': 'probe', 'portsweep': 'probe', 'satan': 'probe',
    'mscan': 'probe', 'saint': 'probe',
    'ftp_write': 'r2l', 'guess_passwd': 'r2l', 'imap': 'r2l', 'multihop': 'r2l',
    'phf': 'r2l', 'spy': 'r2l', 'warezclient': 'r2l', 'warezmaster': 'r2l',
    'sendmail': 'r2l', 'named': 'r2l', 'snmpgetattack': 'r2l', 'snmpguess': 'r2l',
    'xlock': 'r2l', 'xsnoop': 'r2l', 'httptunnel': 'r2l',
    'buffer_overflow': 'u2r', 'loadmodule': 'u2r', 'perl': 'u2r', 'rootkit': 'u2r',
    'ps': 'u2r', 'sqlattack': 'u2r', 'xterm': 'u2r',
}
CLASS_NAMES = ['normal', 'dos', 'probe', 'r2l', 'u2r']
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
CATEGORICAL_COLUMNS = (1, 2, 3)


def read_nsl_kdd(path):
    features, labels = [], []
    with path.open(newline='') as handle:
        for row in csv.reader(handle):
            raw_label = row[41].rstrip('.')
            if raw_label not in ATTACK_GROUPS:
                raise ValueError(f'Unknown NSL-KDD label: {raw_label}')
            features.append(row[:41])
            labels.append(CLASS_TO_INDEX[ATTACK_GROUPS[raw_label]])
    return np.asarray(features, dtype=object), np.asarray(labels, dtype=int)

raw_train, y_train_idx = read_nsl_kdd(NSL_KDD_ROOT / 'KDDTrain%2B.txt')
raw_test, y_test_idx = read_nsl_kdd(NSL_KDD_ROOT / 'KDDTest%2B.txt')
raw_train.shape, raw_test.shape, np.bincount(y_train_idx, minlength=5)

In [ ]:
counts = np.bincount(y_train_idx, minlength=NUM_CLASSES)
plt.figure(figsize=(8, 4))
plt.bar(CLASS_NAMES, counts)
plt.title("NSL-KDD - distribution des classes")
plt.ylabel("Nombre d'exemples")
plt.show()


But : préparer NSL-KDD avec une séparation train/validation/test, encoder les variables catégorielles et normaliser uniquement avec les statistiques du train.

In [ ]:
NUM_CLASSES = 5
SEED = 42
VALIDATION_FRACTION = 0.2
DROPOUT_RATE = 0.01

rng = np.random.default_rng(SEED)
permutation = rng.permutation(len(raw_train))
validation_size = int(len(raw_train) * VALIDATION_FRACTION)
validation_indices = permutation[:validation_size]
train_indices = permutation[validation_size:]

# Encodage one-hot appris uniquement sur le train pour éviter toute fuite.
category_maps = []
for column in CATEGORICAL_COLUMNS:
    category_maps.append({value: index for index, value in enumerate(np.unique(raw_train[train_indices, column]))})


def encode_features(raw_features, fit=False):
    encoded_columns = []
    for column in range(raw_features.shape[1]):
        values = raw_features[:, column]
        if column in CATEGORICAL_COLUMNS:
            mapping = category_maps[CATEGORICAL_COLUMNS.index(column)]
            unknown = len(mapping)
            encoded = np.zeros((len(values), unknown + 1), dtype=np.float32)
            for row, value in enumerate(values):
                encoded[row, mapping.get(value, unknown)] = 1.0
            encoded_columns.append(encoded)
        else:
            encoded_columns.append(values.astype(np.float32)[:, None])
    return np.concatenate(encoded_columns, axis=1)

X_all_train = encode_features(raw_train)
X_test = encode_features(raw_test)
X_train = X_all_train[train_indices]
X_validation = X_all_train[validation_indices]
y_train = np.eye(NUM_CLASSES, dtype=np.float32)[y_train_idx[train_indices]]
y_validation = np.eye(NUM_CLASSES, dtype=np.float32)[y_train_idx[validation_indices]]
y_test = np.eye(NUM_CLASSES, dtype=np.float32)[y_test_idx]

mean = X_train.mean(axis=0, keepdims=True)
std = X_train.std(axis=0, keepdims=True) + 1e-8
X_train = (X_train - mean) / std
X_validation = (X_validation - mean) / std
X_test = (X_test - mean) / std

assert set(train_indices).isdisjoint(validation_indices)
assert X_train.shape[1] == X_validation.shape[1] == X_test.shape[1]
assert y_train.shape[1] == y_validation.shape[1] == y_test.shape[1] == NUM_CLASSES
assert np.array_equal(np.unique(y_train_idx), np.arange(NUM_CLASSES))
assert np.array_equal(np.unique(y_validation_idx), np.arange(NUM_CLASSES))
assert np.array_equal(np.unique(y_test_idx), np.arange(NUM_CLASSES))

INPUT_DIM = X_train.shape[1]
print({'train': X_train.shape, 'validation': X_validation.shape, 'test': X_test.shape,
       'class_names': CLASS_NAMES, 'train_class_counts': np.bincount(y_train_idx[train_indices], minlength=NUM_CLASSES),
       'test_class_counts': np.bincount(y_test_idx, minlength=NUM_CLASSES)})

**But** : reproduire le DNN/MLP de l’article sur NSL-KDD.

Architecture publiée : `41 → 512 → 256 → 128 → 5`, ReLU, dropout `0,01`, focal loss `gamma=2`, `alpha=0,25`.

In [ ]:
HIDDEN_UNITS = (512, 256, 128)
LEARNING_RATE = 1e-4
MOMENTUM = 0.9
L2_STRENGTH = 1e-4
EPOCHS = 250
BATCH_SIZE = 64
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25

model_layers = []
previous_dim = INPUT_DIM
for index, hidden_dim in enumerate(HIDDEN_UNITS):
    model_layers.extend([
        Dense(input_dim=previous_dim, output_dim=hidden_dim, seed=index),
        ReLU(),
        Dropout(rate=DROPOUT_RATE, seed=10 + index),
    ])
    previous_dim = hidden_dim
model_layers.extend([Dense(input_dim=previous_dim, output_dim=NUM_CLASSES, seed=100), Softmax()])
model = Sequential(model_layers)

dense_layers = [layer for layer in model.layers if isinstance(layer, Dense)]
actual_architecture = [dense_layers[0].input_dim, *[layer.output_dim for layer in dense_layers]]
expected_architecture = [INPUT_DIM, *HIDDEN_UNITS, NUM_CLASSES]
assert actual_architecture == expected_architecture
assert model.forward(X_train[:8]).shape == (8, NUM_CLASSES)

loss = MulticlassFocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)
optimizer = SGD(lr=LEARNING_RATE, momentum=MOMENTUM)
regularizer = L2Regularization(strength=L2_STRENGTH)
print({'architecture': actual_architecture, 'loss': 'focal', 'gamma': FOCAL_GAMMA,
       'alpha': FOCAL_ALPHA, 'learning_rate': LEARNING_RATE, 'epochs': EPOCHS,
       'batch_size': BATCH_SIZE, 'note': 'SGD remplace Adam dans le framework actuel'})

## Configuration unique du backend

Une seule exécution est lancée : NumPy sur CPU ou CuPy sur GPU selon `BACKEND`.


In [ ]:
import time

from src.core.backend import get_backend

BACKEND = 'gpu'  # Choisir 'cpu' ou 'gpu'.
backend = get_backend(BACKEND)

model_layers = []
previous_dim = INPUT_DIM
for index, hidden_dim in enumerate(HIDDEN_UNITS):
    model_layers.extend([
        Dense(input_dim=previous_dim, output_dim=hidden_dim, seed=index, backend=backend),
        ReLU(backend=backend),
        Dropout(rate=DROPOUT_RATE, seed=10 + index, backend=backend),
    ])
    previous_dim = hidden_dim
model_layers.extend([
    Dense(input_dim=previous_dim, output_dim=NUM_CLASSES, seed=100, backend=backend),
    Softmax(backend=backend),
])
model = Sequential(model_layers)
loss = MulticlassFocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA, backend=backend)
regularizer = L2Regularization(strength=L2_STRENGTH)

sample_output = model.forward(X_train[:8])
assert backend.is_array(sample_output)
assert sample_output.shape == (8, NUM_CLASSES)
probability_sums = backend.sum(sample_output, axis=1)
assert backend.scalar(probability_sums.min()) > 0.999
assert backend.scalar(probability_sums.max()) < 1.001
print({'backend': 'CuPy' if backend.is_gpu else 'NumPy', 'device': 'CUDA' if backend.is_gpu else 'CPU',
       'architecture': [INPUT_DIM, *HIDDEN_UNITS, NUM_CLASSES], 'loss': 'focal'})

**But** : entraîner le modèle une seule fois avec le backend choisi.

**Pourquoi** : comparer uniquement les performances du backend explicitement sélectionné.


In [ ]:
start_time = time.perf_counter()
scheduler = ReduceLROnPlateau(factor=0.5, patience=10, min_lr=1e-6)
history = train_multiclass_classifier(
    model=model,
    X=X_train,
    y=y_train,
    loss=loss,
    optimizer=optimizer,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_data=(X_validation, y_validation),
    regularizer=regularizer,
    scheduler=scheduler,
)
backend.synchronize()
elapsed_seconds = time.perf_counter() - start_time

assert len(history) == EPOCHS
assert all(row.validation_loss is not None for row in history)
assert all(row.validation_accuracy is not None for row in history)
assert all(row.weight_norm is not None for row in history)
assert all(row.gradient_norm is not None for row in history)
assert all(row.relu_zero_fraction is not None for row in history)

best_epoch = history.best_epoch
best_metrics = history[best_epoch]
assert 0 <= best_epoch < EPOCHS

print({
    'backend': 'CuPy' if backend.is_gpu else 'NumPy',
    'device': 'CUDA' if backend.is_gpu else 'CPU',
    'elapsed_seconds': round(elapsed_seconds, 2),
    'best_epoch': best_epoch,
    'best_validation_loss': best_metrics.validation_loss,
    'best_validation_accuracy': best_metrics.validation_accuracy,
})


**But** : visualiser la loss et l'accuracy train.

**Pourquoi** : verifier la convergence du MLP avant l'evaluation finale sur le test set.


In [ ]:
epochs = [row.epoch for row in history]
losses = [row.loss for row in history]
accuracies = [row.accuracy for row in history]
validation_losses = [row.validation_loss for row in history]
validation_accuracies = [row.validation_accuracy for row in history]
weight_norms = [row.weight_norm for row in history]
gradient_norms = [row.gradient_norm for row in history]
relu_zero_fractions = [row.relu_zero_fraction for row in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, losses)
axes[0].plot(epochs, validation_losses)
axes[0].legend(['train', 'validation'])
axes[0].set_title('Loss (train / validation)')
axes[0].set_xlabel('epoch')
axes[0].grid(True, alpha=0.25)

axes[1].plot(epochs, accuracies)
axes[1].plot(epochs, validation_accuracies)
axes[1].legend(['train', 'validation'])
axes[1].set_title('Accuracy (train / validation)')
axes[1].set_xlabel('epoch')
axes[1].grid(True, alpha=0.25)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, weight_norms)
axes[0].set_title('Weight norm')
axes[1].plot(epochs, gradient_norms)
axes[1].set_title('Gradient norm')
axes[2].plot(epochs, relu_zero_fractions)
axes[2].set_title('ReLU zero fraction')
for ax in axes:
    ax.set_xlabel('epoch')
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


**But** : évaluer le MLP focal loss sur NSL-KDD après la sélection par validation.

In [ ]:
# Evaluation finale sur le test set, sans sélectionner de modèle avec le test.
from src.metrics import multiclass_accuracy

test_probs = model.predict_proba(X_test)
test_probs_host = backend.to_host(test_probs)
test_pred_idx = np.argmax(test_probs_host, axis=1)
test_true_idx = np.argmax(y_test, axis=1)

print({
    'test_accuracy': round(multiclass_accuracy(y_test, test_probs, backend=backend), 4),
    'macro_precision': round(precision_score(test_true_idx, test_pred_idx, average='macro', zero_division=0), 4),
    'macro_recall': round(recall_score(test_true_idx, test_pred_idx, average='macro', zero_division=0), 4),
    'macro_f1': round(f1_score(test_true_idx, test_pred_idx, average='macro', zero_division=0), 4),
})
print('Test set was not used for epoch or hyperparameter selection.')

In [ ]:
cm = confusion_matrix(test_true_idx, test_pred_idx)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES).plot(
    ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False
)
plt.show()

### Résultats finaux NSL-KDD

Le test est évalué après la sélection par validation. La métrique principale est le macro-F1, car le dataset est fortement déséquilibré.

In [ ]:
test_metrics = evaluate_multiclass(model, X_test, y_test, loss)
print({
    'backend': 'CuPy' if backend.is_gpu else 'NumPy',
    'device': 'CUDA' if backend.is_gpu else 'CPU',
    'test_loss': test_metrics.loss,
    'test_accuracy': test_metrics.accuracy,
    'macro_f1': f1_score(test_true_idx, test_pred_idx, average='macro', zero_division=0),
})
print('Le test n’a pas été utilisé pour sélectionner le modèle.')

**But** : afficher la matrice de confusion.

**Pourquoi** : voir quelles classes le MLP confond le plus.
